# Practical 03a — A Rule-Based Knowledge System (Step-by-Step Edition)

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

**Week:** 3 · **Related lectures:** 3, 5, 6, 7, 8 · **Duration:** 3 hours

Student: Muhammad Maaz (2025-cse-027)

## 3. Setup

In [ ]:
___ = None          # the blank you will replace with your answers

def check(label, got, expected, hint=''):
    """Print OK if got == expected, otherwise NOT YET with a hint."""
    if got == expected:
        print('OK      ', label)
    else:
        print('NOT YET ', label)
        print('          expected:', repr(expected))
        print('          you gave:', repr(got))
        if hint:
            print('          hint    :', hint)

print('Setup done. You are ready to start.')

## 4. Python warm-up

### Warm-up 1 — Tuples, and unpacking them

In [ ]:
rule = ('eligible', ['registered', 'attendance_ok'])

print(rule[0])            # the head   (position 0)
print(rule[1])            # the body   (position 1)

head, body = rule         # unpacking: head gets rule[0], body gets rule[1]
print('head =', head)
print('body =', body)

In [ ]:
rule2 = ('hostel_allowed', ['eligible', 'dues_cleared'])
head2, body2 = rule2
print(head2, '<-', body2)

In [ ]:
# CHECK warm-up 1
check('head2', head2, 'hostel_allowed', 'write: head2, body2 = rule2')
check('body2', body2, ['eligible', 'dues_cleared'], 'write: head2, body2 = rule2')

### Warm-up 2 — Sets: `add` and `in`

In [ ]:
known = set(['registered'])     # a set holding one atom
known.add('attendance_ok')
known.add('attendance_ok')      # adding it again changes nothing - no repeats
print(known)
print('registered' in known)    # True
print('eligible' in known)      # False

In [ ]:
known.add('dues_cleared')
is_there = 'dues_cleared' in known
print(known, is_there)

In [ ]:
# CHECK warm-up 2
check("'dues_cleared' added", 'dues_cleared' in known, True, "known.add('dues_cleared')")
check('is_there', is_there, True, "write: 'dues_cleared' in known")

### Warm-up 3 — `all()`

In [ ]:
print(all([True, True, True]))     # True
print(all([True, False, True]))    # False - one is not true

known = {'registered', 'attendance_ok'}
body  = ['registered', 'attendance_ok']
print(all(b in known for b in body))                   # every condition known -> True
print(all(b in known for b in ['registered', 'dues_cleared']))   # dues_cleared missing -> False

In [ ]:
# a) all(b in known for b in ['registered'])
# b) all(b in known for b in ['eligible', 'registered'])
# c) all(b in known for b in [])            <- careful!
pred_a = 'true'
pred_b = False
pred_c = True

In [ ]:
# CHECK warm-up 3
check('a)', pred_a, all(b in known for b in ['registered']), "'registered' is in known")
check('b)', pred_b, all(b in known for b in ['eligible', 'registered']),
      "'eligible' is NOT in known, so not all are true")
check('c)', pred_c, all(b in known for b in []),
      'an empty list has nothing that fails, so all([]) is True')

### Warm-up 4 — `while True`

In [ ]:
n = 40
while True:
    changed = False
    if n % 2 == 0:          # even? then halve it
        n = n // 2
        changed = True
        print('halved ->', n)
    if not changed:         # this round changed nothing -> stop
        break
print('stopped at', n)

In [ ]:
stops_at = 3

In [ ]:
# CHECK warm-up 4
check('24 halved until odd', stops_at, 3, '24 -> 12 -> 6 -> 3, and 3 is odd')

### Warm-up 5 — Recursion

In [ ]:
def countdown(n):
    if n == 0:                 # the stopping case
        return 'lift off'
    print(n)
    return countdown(n - 1)    # the function calls itself with a smaller n

print(countdown(3))

def no_stop(n):
    return no_stop(n - 1)      # no stopping case!

try:
    no_stop(3)
except RecursionError:
    print('no_stop never finished -> RecursionError')

In [ ]:
def count_items(items):
    if items == []:            # stopping case: an empty list has ...
        return 0
    return 1 + count_items(items[1:])    # one item, plus the count of the rest

print(count_items(['a', 'b', 'c']))

In [ ]:
# CHECK warm-up 5
check('count of 3 items', count_items(['a', 'b', 'c']), 3, 'an empty list has 0 items: return 0')
check('count of 0 items', count_items([]), 0, 'return 0')

## 5. Procedure

### Step 1 — Rules and facts as data

In [ ]:
# MEANINGS - written before any rule, as Lecture 4 insisted
#   registered      : enrolled in this course this semester
#   attendance_ok   : attendance is at least 75 percent
#   dues_cleared    : all fees for this semester are paid
#   eligible        : the student may sit the examination
#   hostel_allowed  : the student may stay in the hostel
#   library_card    : the student may be issued a library card

# A rule is (head, [body atoms]).
RULES = [
    ('eligible',       ['registered', 'attendance_ok']),
    ('hostel_allowed', ['eligible', 'dues_cleared']),
]
FACTS = ['registered', 'attendance_ok', 'dues_cleared']

for head, body in RULES:
    print('%-16s <- %s' % (head, ' AND '.join(body)))
print('facts:', FACTS)

In [ ]:
library_rule = ('library_card', ['registered', 'dues_cleared'])
RULES = RULES[:2] + [library_rule]       # the two rules above, plus yours (safe to re-run)

for head, body in RULES:
    print('%-16s <- %s' % (head, ' AND '.join(body)))

In [ ]:
# CHECK Step 1
check('the library rule', library_rule, ('library_card', ['registered', 'dues_cleared']),
      "head first, then the list of conditions: ('library_card', ['registered', 'dues_cleared'])")
check('RULES now has 3 rules', len(RULES), 3, 'run the Step 1 cells again, in order')

### Step 2 — Forward chaining

In [ ]:
def forward_chain(rules, facts, verbose=True):
    known = set(facts)                              # 1. start with the facts
    pass_number = 0
    while True:
        pass_number = pass_number + 1
        added = False                               # nothing new yet in this pass
        for head, body in rules:                    # 2. look at every rule
            if head not in known and all(b in known for b in body):
                known.add(head)                     #    blank 1
                added = True                        #    blank 2
                if verbose:
                    print('pass %d: derived %-16s from %s' % (pass_number, head, body))
        if not added:                               # 4. a whole pass added nothing
            if verbose:
                print('pass %d: nothing new -> stop' % pass_number)
            return known                            #    blank 3

derived = forward_chain(RULES, FACTS)
print('\nfinal derived set:', sorted(derived))

In [ ]:
FACTS2 = ['registered', 'dues_cleared']            # no attendance_ok this time

# Your prediction - a set, e.g. {'a', 'b'}
predicted = {'registered','dues_cleared','library_card'}

print('actual:', sorted(forward_chain(RULES, FACTS2, verbose=False)))

In [ ]:
# CHECK Step 2
check('full fact set', forward_chain(RULES, FACTS, verbose=False),
      {'registered', 'attendance_ok', 'dues_cleared', 'eligible', 'hostel_allowed', 'library_card'},
      'blank 1 is head, blank 2 is True, blank 3 is known')
check('prediction without attendance_ok', predicted, forward_chain(RULES, FACTS2, verbose=False),
      'no attendance_ok -> not eligible -> no hostel; but library_card needs only registered + dues_cleared')

### Step 3 — Backward chaining

In [ ]:
def prove(rules, facts, goal):
    """True if goal can be proved from the facts and rules, otherwise False."""
    if goal in facts:
        return True                                 # blank 1: a fact needs no proof
    for head, body in rules:
        if head == goal:                            # blank 2
            if all(prove(rules, facts, b) for b in body):    # prove every condition
                return True
    return False                                # blank 3

print(prove(RULES, FACTS, 'hostel_allowed'))
print(prove(RULES, FACTS2, 'hostel_allowed'))

In [ ]:
# CHECK Step 3 (plain version)
check('hostel_allowed with FACTS', prove(RULES, FACTS, 'hostel_allowed'), True,
      'blank 1 is True, blank 2 is goal')
check('hostel_allowed with FACTS2', prove(RULES, FACTS2, 'hostel_allowed'), False,
      'blank 3 is False: no rule could prove it')
check('a fact', prove(RULES, FACTS, 'registered'), True, 'blank 1: a fact is proved at once')

In [ ]:
def backward_chain(rules, facts, goal, proof=None, depth=0, seen=None):
    if proof is None:
        proof = {}
    if seen is None:
        seen = set()
    indent = '  ' * depth
    print('%sgoal: %s' % (indent, goal))
    if goal in facts:
        print('%s  -> known fact' % indent)
        proof[goal] = 'fact'
        return True, proof
    if goal in seen:                       # loop guard
        print('%s  -> already being proved: giving up' % indent)
        return False, proof
    seen = seen | {goal}
    for head, body in rules:
        if head == goal:
            print('%s  trying rule %s <- %s' % (indent, head, body))
            if all(backward_chain(rules, facts, b, proof, depth + 2, seen)[0]
                   for b in body):
                proof[goal] = body
                return True, proof
    print('%s  -> cannot establish %s' % (indent, goal))
    return False, proof

ok, proof = backward_chain(RULES, FACTS, 'hostel_allowed')
print('\nproved:', ok)
print('proof :', proof)

In [ ]:
proof_of_eligible = ['registered', 'attendance_ok']      # what is proof['eligible']?
proof_of_registered = 'fact'                             # what is proof['registered']?

In [ ]:
# CHECK Step 3 (proof record)
check("proof['eligible']", proof_of_eligible, proof['eligible'],
      'the body of the rule that proved eligible')
check("proof['registered']", proof_of_registered, proof['registered'],
      "facts are recorded as the string 'fact'")

### Step 4 — Explanation

In [ ]:
def why(proof, atom):
    if atom not in proof:
        return '%s was never established.' % atom
    if proof[atom] == 'fact':
        return '%s was given as a fact.' % atom
    return '%s because %s.' % (atom, ' and '.join(proof[atom]))

for atom in ['hostel_allowed', 'eligible', 'registered']:
    print(why(proof, atom))

In [ ]:
predicted_why = 'library_card was never established.'

In [ ]:
# CHECK Step 4
check('why library_card', predicted_why, why(proof, 'library_card'),
      'backward chaining for hostel_allowed never touched library_card, so it is not in proof')

## 6. Tasks (marked)

### Task 1 — Model your own domain  (5 marks)

In [ ]:
# MEANINGS - every atom, before any rule
#   member            : holds a current library membership for this semester
#   no_dues           : owes no outstanding fine
#   book_on_shelf     : the requested title is physically present in the library
#   lending_copy      : this copy belongs to the lending collection, not the reference one
#   senior_student    : enrolled in year 3 or year 4
#   lab_inducted      : has completed the computing-lab induction
#   printing_quota    : has print credit remaining
#   may_borrow        : permitted to borrow the title at the desk
#   may_take_home     : permitted to remove the copy from the building
#   extended_loan     : entitled to a 28-day loan rather than 14
#   quiet_room_pass   : permitted to book a quiet study room
#   may_reserve       : permitted to place a hold on a title that is out
#   may_use_lab       : permitted to use the computing lab
#   may_borrow_laptop : permitted to borrow a laptop within the building
#   overnight_laptop  : permitted to keep a borrowed laptop overnight
#   may_print         : permitted to send a job to the printers

MY_RULES = [
    ('may_borrow',      ['member', 'no_dues', 'book_on_shelf']),     # rule 1
    ('may_take_home',   ['may_borrow', 'lending_copy']),             # rule 2
    ('extended_loan',   ['may_take_home', 'senior_student']),        # rule 3
    ('quiet_room_pass', ['member', 'no_dues']),                      # rule 4
    ('may_reserve',     ['member', 'no_dues']),                      # rule 5
    ('may_use_lab',     ['member', 'lab_inducted']),                 # rule 6
    ('may_borrow_laptop', ['may_use_lab', 'no_dues']),               # rule 7
    ('overnight_laptop', ['may_borrow_laptop', 'senior_student']),   # rule 8
    ('may_print',       ['may_use_lab', 'printing_quota']),          # rule 9
]
MY_FACTS = ['member', 'no_dues', 'book_on_shelf', 'lending_copy', 'senior_student']

print('%d rules, %d facts' % (len(MY_RULES), len(MY_FACTS)))
for head, body in MY_RULES:
    print('  %-18s <- %s' % (head, ' AND '.join(body)))
print('  facts:', MY_FACTS)

In [ ]:
my_derived = forward_chain(MY_RULES, MY_FACTS, verbose=False)
print('everything that follows:', sorted(my_derived))
print('rule heads NOT derived  :', sorted({h for h, _ in MY_RULES} - my_derived))

In [ ]:
# CHECK Task 1
def _task1():
    ok_shape = isinstance(MY_RULES, list) and all(
        isinstance(r, tuple) and len(r) == 2 and isinstance(r[0], str)
        and isinstance(r[1], list) and all(isinstance(b, str) for b in r[1])
        for r in MY_RULES)
    check('every rule looks like (head, [conditions])', ok_shape, True,
          "each rule is ('head', ['a', 'b']) - a string, then a list of strings")
    check('at least 8 rules', len(MY_RULES) >= 8, True, 'fill in rules 5 to 9')
    check('at least 5 facts', isinstance(MY_FACTS, list) and len(MY_FACTS) >= 5, True,
          'MY_FACTS is a list of the atoms that are true for this student')
    if not ok_shape or not isinstance(MY_FACTS, list):
        return
    got = forward_chain(MY_RULES, MY_FACTS, verbose=False)
    heads = {h for h, _ in MY_RULES}
    check('some rule fires', len(heads & got) > 0, True,
          'at least one rule should be able to fire from your facts')
    check('some rule cannot fire (needed in Task 3)', len(heads - got) > 0, True,
          'leave out a fact so that at least one conclusion fails')
_task1()

### Task 2 — Compare the two procedures  (4 marks)

In [ ]:
def forward_chain_counted(rules, facts):
    known = set(facts)
    examined = 0
    while True:
        added = False
        for head, body in rules:
            examined = examined + 1            # blank 1: one more rule looked at
            if head not in known and all(b in known for b in body):
                known.add(head)
                added = True
        if not added:
            return known, examined

def backward_chain_counted(rules, facts, goal, seen=None, counter=None):
    if counter is None:
        counter = {'n': 0}      # a dictionary, so every recursive call shares ONE counter
    if seen is None:
        seen = set()
    if goal in facts:
        return True, counter['n']
    if goal in seen:
        return False, counter['n']
    seen = seen | {goal}
    for head, body in rules:
        counter['n'] = counter['n'] + 1        # blank 2: one more rule looked at
        if head == goal:
            if all(backward_chain_counted(rules, facts, b, seen, counter)[0] for b in body):
                return True, counter['n']
    return False, counter['n']

In [ ]:
QUERIES = ['quiet_room_pass', 'extended_loan', 'may_print', 'overnight_laptop']

print('%-18s %8s %10s %10s' % ('query', 'proved?', 'forward', 'backward'))
results = {}
for goal in QUERIES:
    known, f_count = forward_chain_counted(MY_RULES, MY_FACTS)
    ok, b_count = backward_chain_counted(MY_RULES, MY_FACTS, goal)
    results[goal] = (ok, f_count, b_count)
    print('%-18s %8s %10d %10d' % (goal, ok, f_count, b_count))

In [ ]:
# 1. Did the FORWARD count change from one query to another?  True or False
forward_changes = False

# 2. For the FIRST query in your list, which examined fewer rules: 'forward' or 'backward'?
fewer_first = 'backward'

# 3. For the LAST query in your list, which examined fewer rules: 'forward' or 'backward'?
fewer_last = 'forward'

In [ ]:
# CHECK Task 2
def _ref_forward(rules, facts):
    known, n = set(facts), 0
    while True:
        added = False
        for h, body in rules:
            n += 1
            if h not in known and all(b in known for b in body):
                known.add(h); added = True
        if not added:
            return n

def _ref_backward(rules, facts, goal, seen=frozenset(), c=None):
    c = c if c is not None else [0]
    if goal in facts: return True, c[0]
    if goal in seen:  return False, c[0]
    for h, body in rules:
        c[0] += 1
        if h == goal and all(_ref_backward(rules, facts, b, seen | {goal}, c)[0] for b in body):
            return True, c[0]
    return False, c[0]

try:
    check('forward count', forward_chain_counted(MY_RULES, MY_FACTS)[1],
          _ref_forward(MY_RULES, MY_FACTS), 'blank 1: examined + 1')
    for g in QUERIES:
        check('backward count for ' + g, backward_chain_counted(MY_RULES, MY_FACTS, g)[1],
              _ref_backward(MY_RULES, MY_FACTS, g)[1], "blank 2: counter['n'] + 1")
    fwd = {results[g][1] for g in QUERIES}
    check('1. does forward change?', forward_changes, len(fwd) > 1,
          'look at the forward column - is every number the same?')
    def fewer(g):
        ok, f, b = results[g]
        return 'backward' if b < f else 'forward'
    check('2. first query', fewer_first, fewer(QUERIES[0]), 'compare the two numbers on the first row')
    check('3. last query', fewer_last, fewer(QUERIES[-1]), 'compare the two numbers on the last row')
except Exception as err:
    print('NOT YET  an error:', type(err).__name__, '-', err, '(usually a blank still ___)')

### Task 3 — Answer a why-not question  (5 marks)

In [ ]:
def provable(rules, facts, goal, seen=frozenset()):
    """Quiet backward chaining: True or False, with a loop guard."""
    if goal in facts:
        return True
    if goal in seen:
        return False
    for head, body in rules:
        if head == goal and all(provable(rules, facts, b, seen | {goal}) for b in body):
            return True
    return False

def why_not(rules, facts, goal):
    """Report which conditions of which rules could not be established.
    Returns a list of (rule, missing_conditions)."""
    if provable(rules, facts, goal):
        print(goal, 'CAN be established - nothing is missing.')
        return []
    report = []
    for head, body in rules:
        if head == goal:                                    # blank 1
            missing = []
            for b in body:
                if not provable(rules, facts, b):           # blank 2
                    missing.append(b)                       # blank 3
            print('rule  %s <- %s' % (head, ' AND '.join(body)))
            print('      MISSING:', missing)
            report.append(((head, body), missing))
    if not report:
        print(goal, 'is not a fact, and no rule concludes it.')
    return report

FAILING_GOAL = 'may_print'          # own domain? put one of your failing atoms here
print('Why not %s?' % FAILING_GOAL)
answer = why_not(MY_RULES, MY_FACTS, FAILING_GOAL)

In [ ]:
def why_not_deep(rules, facts, goal, depth=0, seen=frozenset()):
    pad = '    ' * depth
    rules_for_goal = [(h, b) for h, b in rules if h == goal]
    if not rules_for_goal:
        print(pad + '%s: not a fact, and no rule concludes it  <-- ROOT CAUSE' % goal)
        return
    for head, body in rules_for_goal:
        missing = [b for b in body if not provable(rules, facts, b, seen | {goal})]
        print(pad + '%s <- %s   missing: %s' % (head, ' AND '.join(body), missing))
        for m in missing:
            if m not in seen:
                why_not_deep(rules, facts, m, depth + 1, seen | {goal})

for goal in [FAILING_GOAL, 'overnight_laptop']:
    print('=== why not %s? ===' % goal)
    why_not_deep(MY_RULES, MY_FACTS, goal)
    print()

In [ ]:
ROOT_CAUSE = 'lab_inducted'

In [ ]:
# CHECK Task 3
try:
    check('FAILING_GOAL really fails', provable(MY_RULES, MY_FACTS, FAILING_GOAL), False,
          'choose a goal that forward chaining did NOT derive in Task 1')
    expected = []
    for h, body in MY_RULES:
        if h == FAILING_GOAL:
            expected.append(((h, body), [b for b in body if not provable(MY_RULES, MY_FACTS, b)]))
    check('why_not report', answer, expected, 'blank 1 is goal, blank 2 is provable(rules, facts, b), blank 3 is b')
    heads = {h for h, _ in MY_RULES}
    check('root cause is a condition no rule or fact supplies',
          isinstance(ROOT_CAUSE, str) and ROOT_CAUSE not in MY_FACTS and ROOT_CAUSE not in heads
          and any(ROOT_CAUSE in body for _, body in MY_RULES), True,
          'look for the line marked <-- ROOT CAUSE')
except Exception as err:
    print('NOT YET  an error:', type(err).__name__, '-', err, '(usually a blank still ___)')

**Important:** the system says `lab_inducted` is missing. That could mean the student really has not done the induction, or that nobody has typed the record in yet. "Not proved" is not the same as "false".

### Task 4 — Diagnosis with assumables  (5 marks)

In [ ]:
#   ok_scanner           : ASSUMABLE - the barcode scanner at the desk reads correctly
#   ok_network           : ASSUMABLE - the desk's link to the catalogue server is up
#   issue_recorded       : the catalogue shows this copy as issued
#   still_shows_on_shelf : OBSERVED  - the catalogue shows this copy as available
#   false                : a contradiction

ASSUMABLES = ['ok_scanner', 'ok_network']

DIAG_RULES = MY_RULES + [
    # prediction: a permitted borrower at a working desk -> the loan is recorded
    ('issue_recorded', ['may_take_home', 'ok_scanner', 'ok_network']),
    # contradiction: the loan is recorded AND the book still shows on the shelf
    ('false', ['issue_recorded', 'still_shows_on_shelf']),
]
OBSERVATION = ['still_shows_on_shelf']

def derives_false(assumed):
    """Assume these assumables, add the observation, and see if 'false' follows."""
    facts = MY_FACTS + OBSERVATION + list(assumed)
    return 'false' in forward_chain(DIAG_RULES, facts, verbose=False)

In [ ]:
PREDICT = {
    'assume nothing':          False,
    'assume only ok_scanner':  False,
    'assume only ok_network':  False,
    'assume both':             True,
}

CASES = {
    'assume nothing':          [],
    'assume only ok_scanner':  ['ok_scanner'],
    'assume only ok_network':  ['ok_network'],
    'assume both':             ['ok_scanner', 'ok_network'],
}
print('%-24s %-10s %-10s' % ('case', 'you said', 'actual'))
for name, assumed in CASES.items():
    print('%-24s %-10s %-10s' % (name, PREDICT[name], derives_false(assumed)))

In [ ]:
from itertools import combinations

def find_conflicts(assumables):
    conflicts = []
    for size in range(len(assumables) + 1):              # groups of size 0, 1, 2, ...
        for group in combinations(assumables, size):
            already_covered = any(c <= set(group) for c in conflicts)   # a smaller one already found?
            if derives_false(group) and not already_covered:
                conflicts.append(set(group))
    return conflicts

def find_diagnoses(conflicts, assumables):
    diagnoses = []
    for size in range(len(assumables) + 1):
        for group in combinations(assumables, size):
            hits_every_conflict = all(set(group) & c for c in conflicts)
            already_covered = any(d <= set(group) for d in diagnoses)
            if hits_every_conflict and not already_covered:
                diagnoses.append(set(group))
    return diagnoses

conflicts = find_conflicts(ASSUMABLES)
diagnoses = find_diagnoses(conflicts, ASSUMABLES)
print('conflicts :', [sorted(c) for c in conflicts])
print('diagnoses :', [sorted(d) for d in diagnoses])
for d in diagnoses:
    print('   possible fault:', sorted(d))

In [ ]:
MY_CONFLICT = {'ok_scanner', 'ok_network'}
MY_DIAGNOSES = [{'ok_scanner'}, {'ok_network'}]

In [ ]:
# CHECK Task 4
try:
    check('contradiction rule', DIAG_RULES[-1], ('false', ['issue_recorded', 'still_shows_on_shelf']),
          "('false', ['issue_recorded', 'still_shows_on_shelf'])")
    for name, assumed in CASES.items():
        check('Part A: ' + name, PREDICT[name], derives_false(assumed),
              'false needs issue_recorded, which needs BOTH ok_scanner and ok_network')
    check('Part C: the conflict', [MY_CONFLICT], conflicts, 'the one conflict printed above, as a set')
    _key = lambda s: sorted(s)
    check('Part C: the diagnoses',
          sorted(MY_DIAGNOSES, key=_key) if isinstance(MY_DIAGNOSES, list) else MY_DIAGNOSES,
          sorted(diagnoses, key=_key), 'a list of two one-item sets')
except Exception as err:
    print('NOT YET  an error:', type(err).__name__, '-', err, '(usually a blank still ___)')

### Task 5 — Break it deliberately  (3 marks)

In [ ]:
CYCLIC_RULES = MY_RULES + [
    ('may_print',        ['printer_reserved']),
    ('printer_reserved', ['may_print']),
]

def backward_no_guard(rules, facts, goal):
    """Step 3's prove() - with NO loop guard."""
    if goal in facts:
        return True
    for head, body in rules:
        if head == goal:
            if all(backward_no_guard(rules, facts, b) for b in body):
                return True
    return False

print('1) Backward chaining WITHOUT the loop guard, goal = may_print')
try:
    print('   returned', backward_no_guard(CYCLIC_RULES, MY_FACTS, 'may_print'))
    result_no_guard = 'finished'
except RecursionError:
    print('   RecursionError - it never finished!')
    print('   may_print needs printer_reserved, which needs may_print, which needs ...')
    result_no_guard = 'RecursionError'

print()
print('2) Backward chaining WITH the loop guard (Step 3, unchanged):')
ok, _ = backward_chain(CYCLIC_RULES, MY_FACTS, 'may_print')
print('   returned', ok)

print()
print('3) Forward chaining on the same rules:')
known, examined = forward_chain_counted(CYCLIC_RULES, MY_FACTS)
print('   stopped after examining', examined, 'rules')
print('   may_print derived?       ', 'may_print' in known)
print('   printer_reserved derived?', 'printer_reserved' in known)

In [ ]:
Q1 = 'c'
Q2 = 'b'

In [ ]:
# CHECK Task 5
check('cyclic rules added', CYCLIC_RULES[-2:],
      [('may_print', ['printer_reserved']), ('printer_reserved', ['may_print'])],
      "('may_print', ['printer_reserved']) and ('printer_reserved', ['may_print'])")
check('the run without a guard', result_no_guard, 'RecursionError', 'fill both blanks, then re-run')
check('Q1', Q1, 'c', 'look at what printed under 1)')
check('Q2', Q2, 'b', 'think about Warm-up 4: a loop that stops when a round changes nothing')

**Explanation in my own words:**

*Without the loop guard, backward chaining keeps calling itself again and again and never finishes (python stops it with a RecursionError) because may_print needs printer_reserved and printer_reserved needs may_print again, so it never reaches a fact.*

*Forward chaining does not have this problem because it only ever adds atoms to a set and there are a limited number of atoms, and it stops when a whole pass adds nothing new.*

## 7. Observations

**Rules examined (Task 2)**

| Query | Rules examined (forward) | Rules examined (backward) | Result (proved / not proved) |
|---|---|---|---|
| quiet_room_pass | 18 | 4 | proved |
| extended_loan | 18 | 6 | proved |
| may_print | 18 | 27 | not proved |
| overnight_laptop | 18 | 36 | not proved |

**Why-not (Task 3)**

| Failing goal | Missing condition(s) | Root cause |
|---|---|---|
| may_print | may_use_lab, printing_quota | lab_inducted (and printing_quota) not in the facts |
| overnight_laptop | may_borrow_laptop | lab_inducted |

**Diagnosis (Task 4)**

| Observation that conflicted | Assumables used | Conflict set | Minimal diagnoses |
|---|---|---|---|
| still_shows_on_shelf | ok_scanner, ok_network | {ok_scanner, ok_network} | {ok_scanner} or {ok_network} |

**Cyclic rule (Task 5)**

| Procedure | What happened |
|---|---|
| Backward, no loop guard | RecursionError, it never finished |
| Backward, with loop guard | returned False, it gave up when may_print came back again |
| Forward | stopped normaly, may_print and printer_reserved were not derived |

## 8. Discussion

1. For a query that was proved (like quiet_room_pass) backward chaining examined fewer rules, because it only looks at the rules that the goal leads to, while forward chaining works out everything first so its count is always the same (18 here). But for a query that failed, like overnight_laptop, backward chaining examined more rules than forward chaining, because to be sure it cant be proved it has to try every rule that might conclude it, and then every condition under it too.

2. Reason one: the student really has not done the lab induction (this means it is false). Reason two: the induction was done but nobody entered the record in the facts yet (this means the system simply does not know, so not proved is not the same as false).

3. The rules only describe how the desk works when everything is working correctly. When the observation (book still on the shelf) disagrees with the prediction (loan recorded), at least one assumption (scanner or network ok) must be wrong, so the program finds the faulty part just from the conflict without any rule about breaking.

## 9. Viva questions

(prepare answers before the viva)

## 10. Submission checklist

- [x] Notebook runs top to bottom
- [ ] Every CHECK cell prints OK
- [x] Task 1 has at least eight rules and five facts
- [x] Observations tables filled in
- [x] Task 5 explanation and discussion written
- [x] Name and roll number below
- [x] File named `P03a_2025-cse-027.ipynb`

In [1]:
NAME = "Muhammad Maaz"
ROLL_NUMBER = "2025-cse-027"
print(NAME, ROLL_NUMBER)

Muhammad Maaz 2025-cse-027
